# Fakeddit dataset - Data Exploration Project

## The start


In [ ]:
# Important downloads for on-machine use
%pip install polars
%pip install tqdm
%pip install transformers
%pip install -q -U accelerate bitsandbytes qwen-vl-utils
%pip install --force-reinstall torch torchvision
%pip install sentence-transformers scikit-learn
%pip install lightgbm
%pip install datasketch

# Important downloads for in-cloud use
# %pip install -q kaggle
# %mkdir -p /content/data/fakeddit
# %kaggle datasets download -d vanshikavmittal/fakeddit-dataset -p /content/data/fakeddit --unzip
# %pip install -q -U accelerate bitsandbytes qwen-vl-utils

In [ ]:
# Basic imports
import polars as pl
import matplotlib
import plotly.express as px
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# Advanced imports
import transformers
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
print(transformers.__version__)
print("Qwen2.5-VL - OK")

In [ ]:
# all_data extraction in-cloud
# SAMPLES_DIR = Path("/content/data/fakeddit/multimodal_only_samples")

# all_data: pl.DataFrame = pl.concat([
#     pl.read_csv(SAMPLES_DIR / filename, separator="\t") for filename in SAMPLES_DIR.iterdir()
# ]).with_columns(
#     pl.from_epoch("created_utc", time_unit="s").alias("created_date")
# )

# all_data extraction on-machine
SAMPLES_DIR = Path("multimodal_only_samples") 
all_data: pl.DataFrame = pl.concat([pl.read_csv(filename, separator="\t") for filename in SAMPLES_DIR.iterdir() 
                                    if filename.is_file()]).with_columns(pl.from_epoch("created_utc", time_unit="s").alias("created_date"))

## High Ground EDA of Fakeddit Dataset

### Four basic methods for demostration and description

In [ ]:
all_data.shape

In [ ]:
all_data.schema

In [ ]:
all_data.head()

In [ ]:
all_data.describe()

Dataset consists of nearly 700k rows; each containing of 16 values.

### Main structure of label distribution

In the dataset each row of data is labeled with three types of labels:

**2-way label:** each post is classified as one containing true or false data.

**3-way labels:** posts containing false data have been split into two categories:
  - false text, false sample - sample has `False` label in 2-way labeling and contains untrue information;
  - true text, fale sample - sample has been deemed `False`, but contains "true" text - it involves only posts containing propaganda posters.

**6-way labels:** `True` label remains the same as in the other two kinds of labeling, the `False` label has been split into:
  - imposter content; i.e. content auto-generated by bots;
  - misleading content: content conveyed in a manipulated way to mislead people in believing false narratives;
  - satire/parody;
  - false connection, or the submission images in this category do not accurately support their text descriptions;
  - manipulated content, or the content that has been manually altered e.g. via image modifications.

In [ ]:
# 2-way labels
px.histogram(
    all_data.with_columns(
        label=pl.col("2_way_label").replace_strict({0: "False", 1: "True"}, return_dtype=pl.Utf8)
    ),
    x="label",
    title="Distribution of 2-way labels",
    histnorm="percent",
    text_auto=True,
).show()

In [ ]:
# 3-way labels
px.histogram(
    all_data.with_columns(
        label=pl.col("3_way_label")
        .replace_strict(
            {
                0: "True",
                1: "True text, False sample",
                2: "False text, False sample"
            }
            , return_dtype=pl.Utf8
        )
    ),
    x="label",
    title="Distribution of 3-way labels",
    histnorm="percent",
    text_auto=True,
).show()

In [ ]:
MAP_6_WAY = (
    {
        0: "True",
        1: "Satire / parody",
        2: "False connection",
        3: "Imposter content",
        4: "Manipulated",
        5: "Misleading",
    }
)

px.histogram(
    all_data.with_columns(
        label=pl.col("6_way_label")
        .replace_strict(MAP_6_WAY, return_dtype=pl.Utf8)
    ),
    x="label",
    title="Distribution of 6-way labels",
    histnorm="percent",
    text_auto=True,
).show()

### Further EDA int the context of the project

In [ ]:
titles_length = (
    all_data.select("clean_title")
    .drop_nulls()
    .with_columns(
        pl.col("clean_title")
        .str.split(" ")
        .list.len()
        .alias("word_count")
    )
)

titles_length.head()

In [ ]:
word_count_q99 = titles_length.select(pl.col("word_count").quantile(0.99)).item()

word_count_q99

In [ ]:
world_count_plot = px.histogram(
    titles_length.filter(pl.col('word_count') < word_count_q99),
    x="word_count",
    nbins=int(word_count_q99),
    title="Distribution of cleaned title word counts")

world_count_plot.update_layout(
    xaxis_title="Number of words in title",
    yaxis_title="Count")

world_count_plot.show()

In [ ]:
score_high = all_data.select(pl.col("score").quantile(0.95)).item()
score_low = all_data.select(pl.col("score").quantile(0.05)).item()

px.histogram(
    all_data.filter((score_low <= pl.col("score")) & (pl.col("score")  <= score_high)),
    x="score",
    nbins=100,
    title="Distribution of scores for subreddit posts")

In [ ]:
day_of_post = px.histogram(
    all_data,
    x="created_date",
    nbins=100,
    title="Post distribution over time")

day_of_post.show()

In [ ]:
six_labels_dist = px.histogram(
    all_data.with_columns(
        label=pl.col("6_way_label")
        .replace_strict(MAP_6_WAY, return_dtype=pl.Utf8)),
    x="subreddit",
    title="Distribution of 6-way labels",
    histnorm="percent",
    text_auto=True,
    color="label")

six_labels_dist.show()

**Fundamental conclusions**

The [paper](https://arxiv.org/pdf/1911.03854) that introduced the dataset has discussed the labeling strategy.
For each post in a given subreddit the same label was assigned, i.e. all `r/psbattle_artwork` are labeled as "Manipulated" - since it is a subreddit containing modified images (although they seem to serve satirical purpose rather than misleading one). Given high class imbalance and dubious labeling strategy, we've decided to choose 2-way labels for the classification experiments. We fear that for many cases intentions behind false information has been mislabeled. The 2-way labels - although not semantically rich - seem to be more reliable and trustworthy than 3 or 6-way ones.

## Cleaning The Fakeddit Dataset

For the purpose of making the text data easier to use for downstream tasks, we'll:
- get rid of duplicates (for now we'll test with exact duplicates)
- get rid of subreddit names; although it seems useful - it may introduce data leakage since there's always just one label for each samples associated with a particular subreddit.
- get rid of image urls since these won't be used, same goes `domain` column which identifies the image hosting service.
- handle `null` values in columns

### Finding dublicates and NULL values

THIS TEXT BELOW BE UTILIZED PROBABLY

To check for post duplicates we analyze the text contents using MinHash algorithm. For that purpose we split titles into 3-grams, which were later treated as input for MinHash. The title duplication may not necesarilly indicate post duplication; i.e. two reddit posts may have the same title but different images associated with them. For more reliable results we could introduce similarity metric for linked images (i.e. by computing vector embeddings) - that's being omitted for now though. We will also use the MinHash similarities between titles to get almost-duplicates. These will be posts containing the same photo (indicated by the `image_url` field) with similar titles.

In [ ]:
# Null values in a whole dataframe whole dataframe
nulls_in_data = (
    all_data.null_count()
    .transpose(include_header=True, column_names=["null_count"])
    .with_columns((pl.col("null_count") / all_data.height).alias("null_pct")))
with pl.Config(tbl_rows=-1, tbl_cols=-1): display(nulls_in_data) 

In [ ]:
messy_dubliKates = (
    all_data.group_by(["clean_title"])
      .len()
      .filter(pl.col("len") > 1)
      .sort("len", descending=True))

messy_dubliKates

In [ ]:
exact_dubliKates = (
    all_data.group_by(["clean_title", 'image_url'])
      .len()
      .filter(pl.col("len") > 1)
      .sort("len", descending=True)
)

exact_dubliKates

We've found $3 \ 522$ (almost) duplicate pairs of cleaned post titles. It's relatively small compared to the whole contents of the dataset.
These are unique pairs. There are over 20k titles that have appeared multiple times. Sharing title is not enough to deem posts as exact duplicates, we can use `image_url` for that.

In [ ]:
# Integrated minihash dublicates finding

THRESHOLD = 0.9
NUM_PERM = 64
K = 4

titles_df = all_data.select(pl.col("clean_title")).with_row_index("title_id")
titles = titles_df["clean_title"].to_list()
ids = titles_df["title_id"].to_list()

def shingle_bytes(text: str, k: int = K):
    if len(text) < k:
        yield text.encode("utf-8")
        return
    for i in range(len(text) - k + 1):
        yield text[i:i+k].encode("utf-8")

lsh = MinHashLSH(threshold=THRESHOLD, num_perm=NUM_PERM)
minhash_dict = {}

with lsh.insertion_session() as session:
    for i, title in tqdm(zip(ids, titles), total=len(ids)):
        mh = MinHash(num_perm=NUM_PERM)
        for token in shingle_bytes(title):
            mh.update(token)
        minhash_dict[i] = mh
        session.insert(i, mh)

pairs = []
for i, mh in tqdm(minhash_dict.items(), total=len(minhash_dict)):
    candidates = lsh.query(mh)
    for j in candidates:
        if i >= j:
            continue
        sim = mh.jaccard(minhash_dict[j])
        if sim >= THRESHOLD:
            pairs.append((titles[i], titles[j], sim))

similar_titles_df = (
    pl.DataFrame(pairs, schema=["title_a", "title_b", "similarity"], orient="row")
    .sort("similarity", descending=True)
)
print(f"Znaleziono {len(similar_titles_df)} podobnych par.")

lazy_title_to_image_map = all_data.lazy().select(["clean_title", "image_url"]).unique(subset=["clean_title"])

df_confirmed_duplicates = (
    similar_titles_df.lazy()
    .join(lazy_title_to_image_map, left_on="title_a", right_on="clean_title", how="left")
    .rename({"image_url": "image_url_a"})
    .join(lazy_title_to_image_map, left_on="title_b", right_on="clean_title", how="left")
    .rename({"image_url": "image_url_b"})
    .filter((pl.col("image_url_a") == pl.col("image_url_b")) & (pl.col("image_url_a").is_not_null()))
    .collect()
)

print(f"Potwierdzono {len(df_confirmed_duplicates)} duplikatów ze zgodnym zdjęciem.")
display(df_confirmed_duplicates)

### Preparing cleared data

In [ ]:
trimmed_data = all_data.select(
    pl.col("author"),
    pl.col("created_utc"),
    pl.col("clean_title"),
    pl.col("score"),
    pl.col("2_way_label"),
    pl.col("6_way_label"),
    pl.col("num_comments"),
    pl.col("upvote_ratio"),
    pl.col("image_url"),
).unique(
    subset=["clean_title", "image_url"],
    keep="first"
).drop("image_url")
trimmed_data.describe()

In [ ]:
display(all_data.filter(pl.col("num_comments").is_null()).head())
all_data.filter(pl.col("num_comments").is_null()).select(pl.col("subreddit").n_unique())

Missing values seem to correlate with `r/psbattle_artwork` subreddit. This one does not seem to exist anymore; it seems reasonable to remove it. We'll recreate `trimmed_data` in a messy way.

In [ ]:
trimmed_data = all_data.select(
    pl.col("author"),
    pl.col("created_utc"),
    pl.col("clean_title"),
    pl.col("score"),
    pl.col("2_way_label"),
    pl.col("6_way_label"),
    pl.col("num_comments"),
    pl.col("upvote_ratio"),
    pl.col("image_url"),
).unique(
    subset=["clean_title", "image_url"],
    keep="first"
).drop("image_url")
trimmed_data.describe()

It did indeed fix the `null` situation.

## First regression models (baseline) 

This will be a simple, text-only TF-IDF model based on unigrams and bigrams. Metadata and images (additional modalities in p=other words) will be ommited for this step of project advancement.

In [ ]:
# Additional libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score)
from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from typing import Self

In [ ]:
# Binary Baseline
# 0 (True) remains 0, the rest (1-5) become 1 (Fake)
model_data_bin = (
    trimmed_data
    .select(["clean_title", "6_way_label"])
    .drop_nulls()
    .with_columns(
        pl.when(pl.col("6_way_label") == 0)
        .then(0)
        .otherwise(1)
        .alias("binary_label")
    )
)

X_bin = model_data_bin["clean_title"]
y_bin = model_data_bin["binary_label"]

# Train-test split
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

print("Class distribution (Training set - Binary):")
print(y_train_bin.value_counts(normalize=True))

# Defining and training the binary pipeline
pipeline_bin = Pipeline([
    ("TF-IDF", TfidfVectorizer(ngram_range=(1, 2), max_features=30_000, lowercase=True)),
    ("Logi", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipeline_bin.fit(X_train_bin, y_train_bin)

# Binary predictions
y_pred_bin = pipeline_bin.predict(X_test_bin)
y_proba_bin = pipeline_bin.predict_proba(X_test_bin)[:, 1] # Probability of class 1 (Fake)

# Binary evaluation metrics
roc_auc_bin = roc_auc_score(y_test_bin, y_proba_bin)
print(f"\nROC AUC (Binary): {roc_auc_bin:.4f}\n")
print(classification_report(y_test_bin, y_pred_bin, digits=4, target_names=["True (0)", "Fake (1)"]))

# Visualisation and interpretation

# 1. Confusion matrix visualization
cm_bin = confusion_matrix(y_test_bin, y_pred_bin)
px.imshow(
    cm_bin, text_auto=True, 
    x=["True (0)", "Fake (1)"], y=["True (0)", "Fake (1)"], 
    color_continuous_scale="Reds", 
    labels=dict(x="Predicted label", y="True label", color="Count"),
    title="Confusion Matrix - Binary Model"
).show()

# 2. Extracting the most important words for both binary classes
clf_bin = pipeline_bin.named_steps["Logi"]
vectorizer_bin = pipeline_bin.named_steps["TF-IDF"]
feature_names_bin = np.array(vectorizer_bin.get_feature_names_out())

# In a binary model, we only have one set of weights: coef_[0]
coefs_bin = clf_bin.coef_[0]

TOP_N = 10
top_true_idx = np.argsort(coefs_bin)[:TOP_N]           # Smallest (negative) weights -> True
top_fake_idx = np.argsort(coefs_bin)[-TOP_N:][::-1]    # Largest (positive) weights -> Fake

# Assembling the results into a readable table
top_features_df_bin = pd.DataFrame({
    "Words (True)": feature_names_bin[top_true_idx],
    "Weight (True)": coefs_bin[top_true_idx],
    "Words (Fake)": feature_names_bin[top_fake_idx],
    "Weight (Fake)": coefs_bin[top_fake_idx]
})
top_features_df_bin.index = [f"Rank {i+1}" for i in range(TOP_N)]

print("\nTop terms driving the binary predictions:")
display(top_features_df_bin)

In [ ]:
# 6-labels model
# Comparing text embeddings
model_data = (
    trimmed_data
    .select(["clean_title", "6_way_label"])
    .drop_nulls()
)

display(model_data.shape)

X = model_data["clean_title"]
y = model_data["6_way_label"]

# Added "_multi" suffix so Stage 3 picks this up automatically
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train size:", len(X_train_multi))
print("Test size:", len(X_test_multi))
print("Train label distribution:")
print(y_train_multi.value_counts(normalize=True))

baseline_model = Pipeline([
    (
        "TF-IDF",
        TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=30_000,
            lowercase=True,
        ),
    ),
    (
        "Logi",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

baseline_model.fit(X_train_multi, y_train_multi)

y_pred = baseline_model.predict(X_test_multi)
y_proba = baseline_model.predict_proba(X_test_multi)

roc_auc = roc_auc_score(y_test_multi, y_proba, multi_class="ovr")

print(f"{roc_auc=}")
print(classification_report(y_test_multi, y_pred, digits=4))

# Dictionary with class names
label_names = {
    0: "True",
    1: "Satire / parody",
    2: "False connection",
    3: "Imposter content",
    4: "Manipulated",
    5: "Misleading",
}

clf = baseline_model.named_steps["Logi"]
vectorizer = baseline_model.named_steps["TF-IDF"]

class_labels = clf.classes_
string_labels = [label_names[c] for c in class_labels] # Generating text names

# Quick display of the confusion matrix
cm = confusion_matrix(y_test_multi, y_pred, labels=class_labels)
px.imshow(
    cm, 
    text_auto=True, 
    x=string_labels, 
    y=string_labels, 
    color_continuous_scale="Blues",
    labels=dict(x="Predicted label", y="True label", color="Count"),
    title="Confusion Matrix - Multiclass Baseline"
).show()

# Vectorized feature extraction
TOP_N = 20
feature_names = np.array(vectorizer.get_feature_names_out())

# Getting indices of TOP-N features for each class simultaneously (row - class, columns - indices)
top_indices = np.argsort(clf.coef_, axis=1)[:, -TOP_N:][:, ::-1]

# Creating a dictionary where the key is the class name, the value is the list of words
top_features_dict = {
    string_labels[i]: feature_names[top_indices[i]] 
    for i in range(len(string_labels))
}

# Conversion to a nice table for the report
top_features_df = pd.DataFrame(top_features_dict)
top_features_df.index = [f"Rank {i+1}" for i in range(TOP_N)]

print("Top terms driving the predictions for each class:")
display(top_features_df)

## Semantic Embeddings & Advanced Classifiers

While the baseline models relied on lexical frequency (TF-IDF), this stage leverages deep learning to capture the actual semantic meaning of the titles. 

We will use HuggingFace `SentenceTransformers` to convert text into dense vector representations (embeddings). To evaluate the best approach, we built an automated experimentation pipeline that tests multiple combinations of:
1. **Embedders:** `all-MiniLM-L12-v2` and `paraphrase-MiniLM-L3-v2` (Fast and efficient transformers).
2. **Classifiers:** Logistic Regression, Random Forest, and LightGBM.

This engine will train all combinations, evaluate their performance, log errors for further analysis, and save the artifacts.

In [ ]:
# Transformers for embeddings

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from typing import Self

class TitleEmbedder(TransformerMixin, BaseEstimator):
    """
    A custom scikit-learn transformer that wraps HuggingFace SentenceTransformers.
    This allows us to seamlessly integrate deep learning embeddings into standard ML pipelines.
    """
    def __init__(self, model_name: str, *, batch_size: int = 256) -> None:
        self._model_name = model_name
        self._batch_size = batch_size
        self._model = None # Initialized during fit

    def fit(self, X: np.ndarray, y: None = None) -> Self:
        # Load the pre-trained transformer model
        self._model = SentenceTransformer(self._model_name)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        # Convert text into dense vector embeddings
        return self._model.encode(
            list(X),
            batch_size=self._batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

In [ ]:
# Configuration setup of experiments

import os
import gc
import torch
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, asdict

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

# Define output directories for model artifacts
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "confusion_matrices").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "errors").mkdir(exist_ok=True)

# Re-using the labels from the baseline stage
LABEL_NAMES = ["True", "Satire / parody", "False connection", "Imposter content", "Manipulated", "Misleading"]

# Define the Embedders (Transformers) to test
EMBEDDINGS = {
    "minilm-really": ("emb", TitleEmbedder("all-MiniLM-L12-v2")),
    "paraphrase": ("emb", TitleEmbedder("paraphrase-MiniLM-L3-v2")),
}

# Define the Classifiers to test
CLASSIFIERS = {
    "logreg": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    "rf": lambda: RandomForestClassifier(
        n_estimators=300, max_depth=None, class_weight="balanced", n_jobs=-1, random_state=42
    ),
    "lgbm": lambda: lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.1, class_weight="balanced", num_leaves=63, n_jobs=-1, random_state=42, verbose=-1
    ),
}

# Combine them into a grid of pipelines
MODELS = {}
for emb_name, (step_name, emb_step) in EMBEDDINGS.items():
    for clf_name, clf_factory in CLASSIFIERS.items():
        model_key = f"{emb_name}__{clf_name}"
        MODELS[model_key] = Pipeline([
            (step_name, emb_step),
            ("clf", clf_factory()),
        ])

print(f"Total model configurations to train: {len(MODELS)}")
for k in MODELS:
    print(f"  • {k}")

# Dataclass to store metrics for final comparison
@dataclass
class ModelResult:
    name: str
    roc_auc: float
    f1_macro: float
    accuracy: float
    precision_macro: float
    recall_macro: float

In [ ]:
# Training and evaluation

import matplotlib
import matplotlib.pyplot as plt
import plotly.express as px
from tqdm import tqdm  # Zwykły pasek postępu (bez ipywidgets)
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Use 'Agg' backend to avoid opening multiple blank windows in Jupyter
matplotlib.use("Agg")

results: list[ModelResult] = []
all_errors: list[pd.DataFrame] = []


# --- QUICK TEST MODE (Remove for final run) ---
X_train_multi = X_train_multi[:500]
y_train_multi = y_train_multi[:500]
X_test_multi = X_test_multi[:100]
y_test_multi = y_test_multi[:100]
# ----------------------------------------------

# Assuming X_train_multi, y_train_multi, X_test_multi, y_test_multi are available from Stage 2
for model_name, model in tqdm(MODELS.items(), desc="Evaluating Models"):
    print(f"\n{'='*20} {model_name} {'='*20}")

    # Train the pipeline
    model.fit(X_train_multi, y_train_multi)
    
    # Generate predictions
    y_pred = model.predict(X_test_multi)
    y_proba = model.predict_proba(X_test_multi)

    # Calculate metrics
    roc_auc = roc_auc_score(y_test_multi, y_proba, multi_class="ovr", average="macro")
    report = classification_report(y_test_multi, y_pred, target_names=LABEL_NAMES, digits=4, output_dict=True)

    result = ModelResult(
        name=model_name,
        roc_auc=roc_auc,
        f1_macro=report["macro avg"]["f1-score"],
        accuracy=report["accuracy"],
        precision_macro=report["macro avg"]["precision"],
        recall_macro=report["macro avg"]["recall"],
    )
    results.append(result)
    print(result)

    # --- Generate and Save Static Confusion Matrix (Matplotlib) ---
    cm = confusion_matrix(y_test_multi, y_pred)
    fig_mpl, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues")
    fig_mpl.colorbar(im, ax=ax)

    ax.set_xticks(range(len(LABEL_NAMES)))
    ax.set_yticks(range(len(LABEL_NAMES)))
    ax.set_xticklabels(LABEL_NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(LABEL_NAMES, fontsize=8)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"{model_name} — ROC-AUC: {roc_auc:.4f} | F1 macro: {result.f1_macro:.4f}")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)

    fig_mpl.tight_layout()
    fig_mpl.savefig(ARTIFACTS_DIR / "confusion_matrices" / f"{model_name}.png", dpi=200)
    plt.close(fig_mpl)

    # --- Display Interactive Confusion Matrix (Plotly) ---
    px.imshow(
        cm, x=LABEL_NAMES, y=LABEL_NAMES, text_auto=True,
        color_continuous_scale="Blues",
        labels=dict(x="Predicted label", y="True label", color="Count"),
        title=f"{model_name} — ROC-AUC: {roc_auc:.4f} | F1 macro: {result.f1_macro:.4f}",
        height=600,
    ).show()

    # --- Error Analysis: Save Misclassified Samples ---
    y_test_np = np.array(y_test_multi)
    y_pred_np = np.array(y_pred)
    misclassified_mask = y_test_np != y_pred_np

    if misclassified_mask.any():
        X_test_list = list(X_test_multi)
        error_df = pd.DataFrame({
            "model": model_name,
            "index": np.where(misclassified_mask)[0],
            "text": [X_test_list[i] for i in np.where(misclassified_mask)[0]],
            "true_label": y_test_np[misclassified_mask],
            "true_label_name": [LABEL_NAMES[l] for l in y_test_np[misclassified_mask]],
            "pred_label": y_pred_np[misclassified_mask],
            "pred_label_name": [LABEL_NAMES[l] for l in y_pred_np[misclassified_mask]],
            "pred_proba_max": y_proba[misclassified_mask].max(axis=1),
        })
        error_df.to_csv(ARTIFACTS_DIR / "errors" / f"{model_name}_errors.tsv", sep="\t", index=False)
        all_errors.append(error_df)

    # --- Free Memory ---
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Aggregate and Display Final Results ---
results_df = pd.DataFrame([asdict(r) for r in results]).sort_values("f1_macro", ascending=False)
results_df.to_csv(ARTIFACTS_DIR / "results_table.tsv", sep="\t", index=False)

if all_errors:
    combined_errors = pd.concat(all_errors, ignore_index=True)
    combined_errors.to_csv(ARTIFACTS_DIR / "all_errors_combined.tsv", sep="\t", index=False)
    print(f"\nTotal misclassifications saved across all models: {len(combined_errors)}")

print(f"\nAll artifacts successfully saved to: {ARTIFACTS_DIR.resolve()}")

print("\n--- Final Model Comparison ---")
display(results_df)

## Hyperparameters tuning with Optuna

### Preparation of embeddings and validation

In [ ]:
print("Initializing embedder")
best_embedder = TitleEmbedder("all-MiniLM-L12-v2", batch_size=256)
print("Computing embeddings")

X_train_emb = best_embedder.fit_transform(X_train) 
X_test_emb = best_embedder.transform(X_test)
X_tr_emb, X_val_emb, y_tr, y_val = train_test_split(
    X_train_emb, y_train, 
    test_size=0.15, 
    random_state=777, 
    stratify=y_train
)

print(f"Data shapes - Train: {X_tr_emb.shape}, Val: {X_val_emb.shape}, Test: {X_test_emb.shape}")

### Optuna setup and testing

In [ ]:
def objective(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 6,
        'metric': 'multi_logloss',
        'class_weight': 'balanced',
        'boosting_type': 'gbdt',
        'random_state': 777,
        'n_jobs': -1,
        'verbose': -1,
        'device': 'gpu', 
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr_emb, y_tr,
        eval_set=[(X_val_emb, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=50)
        ]
    )
    
    y_pred_val = model.predict(X_val_emb)
    return f1_score(y_val, y_pred_val, average='macro')

study = optuna.create_study(study_name='lgbm_fakeddit_optimization', direction='maximize')
print("Starting Optuna optimization")
study.optimize(objective, n_trials=20) 

print("Best validation F1-Macro:")
print(f"{study.best_value:.4f}")
print("Best parameters:")
for key, value in study.best_params.items():
    print(f"{key}: {value}")

print("Training final model with best parameters")
best_params = study.best_params
best_params.update({
    'objective': 'multiclass',
    'num_class': 6,
    'class_weight': 'balanced',
    'random_state': 777,
    'n_jobs': -1,
    'verbose': -1,
    'device': 'gpu'
})

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_train_emb, y_train)

y_test_pred = final_model.predict(X_test_emb)
y_test_proba = final_model.predict_proba(X_test_emb)
final_roc_auc = roc_auc_score(y_test, y_test_proba, multi_class="ovr", average="macro")

print(f"Final test ROC-AUC: {final_roc_auc:.4f}")
print("Final Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=LABEL_NAMES, digits=4))

### Post-"Optuna" Analysis

In [ ]:
best_trial = study.best_trial

print(f"Best trial number: {best_trial.number}")
print(f"Best validation F1-Macro: {best_trial.value:.4f}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

best_params = best_trial.params.copy()
best_params.update({
    'objective': 'multiclass',
    'num_class': 6,
    'class_weight': 'balanced',
    'random_state': 777,
    'n_jobs': -1,
    'verbose': -1
})

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_train_emb, y_train)

y_pred_test = final_model.predict(X_test_emb)

final_f1 = f1_score(y_test, y_pred_test, average='macro')
final_acc = accuracy_score(y_test, y_pred_test)

baseline_f1_macro = 0.5654
baseline_accuracy = 0.6765

print(f"Baseline F1-Macro: {baseline_f1_macro:.4f}")
print(f"Optimized F1-Macro: {final_f1:.4f}")
print(f"F1-Macro improvement: {(final_f1 - baseline_f1_macro):.4f}")

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_test,
    display_labels=LABEL_NAMES,
    cmap='Blues',
    ax=ax,
    xticks_rotation=45
)
plt.title('Confusion matrix for optimized model')
plt.tight_layout()
plt.show()

importances = final_model.feature_importances_
feature_names = [f"Embedding_Dim_{i}" for i in range(X_train_emb.shape[1])]

feature_importance_df = pd.DataFrame({
    'Feature': feature_names, 
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(20))
plt.title('Feature importances from final model')
plt.tight_layout()
plt.show()

## VLMS -- need for clarification

In [ ]:
from hashlib import sha256

def url_to_filename(url: str) -> str:
    """Hashes the URL to create a unique, safe filename for the image."""
    return f"{sha256(url.encode('UTF-8')).hexdigest()}.jpg"

# Apply hashing and preview the new filenames (showing last 10 characters for brevity)
image_filenames_df = all_data.with_columns(
    pl.col("image_url")
    .map_elements(lambda url: url_to_filename(url)[-10:], return_dtype=pl.String)
    .alias("hashed_filename")
).select(["image_url", "hashed_filename"])

display(image_filenames_df.head())

In [ ]:
import os

cpu_count = os.cpu_count()
print(cpu_count)

In [ ]:
import time
import threading
from pathlib import Path
from hashlib import sha256
from io import BytesIO
from typing import Literal, TypedDict, Union
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from collections import Counter

import polars as pl
import requests
from requests.adapters import HTTPAdapter
from PIL import Image, ImageOps
from tqdm.auto import tqdm

MAX_WORKERS = 96
MAX_IN_FLIGHT = 512
TIMEOUT = 5
JPEG_QUALITY = 85
MAX_LONG_SIDE = 512
IMAGE_DIR = Path("/kaggle/images")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

_thread_local = threading.local()


class ImageOk(TypedDict):
    status: Literal["ok"]
    image_url: str
    image_path: str


class ImageError(TypedDict):
    status: Literal["error"]
    image_url: str
    exception_type: str
    exception: str
    http_status: int | None


ImageResult = Union[ImageOk, ImageError]


def get_session() -> requests.Session:
    if not hasattr(_thread_local, "session"):
        session = requests.Session()
        session.headers.update({
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) "
                "Gecko/20100101 Firefox/150.0"
            )
        })

        adapter = HTTPAdapter(
            pool_connections=MAX_WORKERS,
            pool_maxsize=MAX_WORKERS,
            max_retries=0,
        )

        session.mount("http://", adapter)
        session.mount("https://", adapter)

        _thread_local.session = session

    return _thread_local.session


def url_to_filename(url: str) -> str:
    return sha256(url.encode("UTF-8")).hexdigest() + ".jpg"


def fetch_and_save_image(url: str) -> ImageResult:
    path = IMAGE_DIR / url_to_filename(url)

    if path.exists():
        return {
            "status": "ok",
            "image_url": url,
            "image_path": str(path),
        }

    tmp_path = path.with_suffix(".jpg.tmp")

    try:
        response = get_session().get(url, timeout=TIMEOUT)

        if response.status_code != 200:
            return {
                "status": "error",
                "image_url": url,
                "exception_type": "HTTPError",
                "exception": f"HTTP status {response.status_code}",
                "http_status": response.status_code,
            }

        img = Image.open(BytesIO(response.content))
        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")

        img.thumbnail(
            (MAX_LONG_SIDE, MAX_LONG_SIDE),
            Image.Resampling.LANCZOS,
        )

        img.save(
            tmp_path,
            format="JPEG",
            quality=JPEG_QUALITY,
            subsampling=2,
            optimize=True,
        )
        tmp_path.replace(path)

        return {
            "status": "ok",
            "image_url": url,
            "image_path": str(path),
        }

    except Exception as e:
        try:
            if tmp_path.exists():
                tmp_path.unlink()
        except Exception:
            pass

        return {
            "status": "error",
            "image_url": url,
            "exception_type": type(e).__name__,
            "exception": repr(e),
            "http_status": None,
        }

In [ ]:
N_SAMPLES = 75_000

urls = (
    all_data
    .select("image_url")
    .drop_nulls()
    .unique()
    .get_column("image_url")
    .to_list()
)[:N_SAMPLES]

results: list[ImageResult] = []
exception_counts = Counter()
http_status_counts = Counter()

urls_iter = iter(urls)
start = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    in_flight = set()

    for _ in range(min(MAX_IN_FLIGHT, len(urls))):
        try:
            url = next(urls_iter)
        except StopIteration:
            break

        in_flight.add(executor.submit(fetch_and_save_image, url))

    with tqdm(total=len(urls)) as pbar:
        while in_flight:
            done, in_flight = wait(in_flight, return_when=FIRST_COMPLETED)

            for future in done:
                result = future.result()
                results.append(result)

                if result["status"] == "error":
                    exception_counts[result["exception_type"]] += 1
                    http_status_counts[result["http_status"]] += 1

                pbar.update(1)

                try:
                    url = next(urls_iter)
                    in_flight.add(executor.submit(fetch_and_save_image, url))
                except StopIteration:
                    pass

elapsed = time.time() - start

total = len(results)
errors = sum(exception_counts.values())
successes = total - errors

print(f"Images attempted: {total}")

if total > 0:
    print(f"Successful images: {successes} ({successes / total:.2%})")
    print(f"Errors: {errors} ({errors / total:.2%})")
    print(f"Elapsed: {elapsed:.1f}s")
    print(f"Images/sec: {total / elapsed:.2f}")

print("\nErrors by exception type:")
for exception_type, count in exception_counts.most_common():
    print(f"{exception_type}: {count}")

print("\nHTTP errors by status code:")
for status, count in http_status_counts.most_common():
    print(f"{status}: {count}")

In [ ]:
import gc
import numpy as np
import polars as pl
import torch
import lightgbm as lgb

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, roc_auc_score

from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

manifest = pl.DataFrame(results)

model_data = (
    all_data
    .unique(subset=["clean_title", "image_url"], keep="first")
    .join(
        manifest
        .filter(pl.col("status") == "ok")
        .select(["image_url", "image_path"]),
        on="image_url",
        how="inner",
    )
    .select(["clean_title", "image_path", "6_way_label"])
    .drop_nulls()
)

pdf = model_data.to_pandas()

X_train, X_test, y_train, y_test = train_test_split(
    pdf[["clean_title", "image_path"]],
    pdf["6_way_label"].astype(int),
    test_size=0.2,
    random_state=42,
    stratify=pdf["6_way_label"],
)


MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)

model.eval()

In [ ]:

def make_message(title: str, image_path: str):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": f"Represent this Reddit post for classification.\nTitle: {title}"},
        ],
    }]


@torch.inference_mode()
def embed_batch(titles, image_paths):
    messages = [make_message(t, p) for t, p in zip(titles, image_paths)]

    texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in messages
    ]

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model(
        **inputs,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )

    h = outputs.hidden_states[-1]
    mask = inputs["attention_mask"].unsqueeze(-1).to(h.dtype)

    emb = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
    emb = emb.float().cpu().numpy()

    del inputs, outputs, h, mask
    gc.collect()
    torch.cuda.empty_cache()

    return emb

def encode_df(df_part, batch_size=2, out_path=None):
    embeddings = []

    titles = df_part["clean_title"].astype(str).tolist()
    paths = df_part["image_path"].astype(str).tolist()

    for i in tqdm(range(0, len(df_part), batch_size)):
        emb = embed_batch(
            titles[i:i + batch_size],
            paths[i:i + batch_size],
        )
        embeddings.append(emb)

    X_emb = np.vstack(embeddings).astype(np.float32)

    if out_path is not None:
        np.save(out_path, X_emb)

    return X_emb

In [ ]:

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
gc.collect()

In [ ]:
N_TRAIN = 128
N_TEST = 64
BATCH_SIZE = 128

X_train_emb = encode_df(
    X_train,
    batch_size=BATCH_SIZE,
    out_path="/content/qwen25vl_train_10k.npy",
)

X_test_emb = encode_df(
    X_test,
    batch_size=BATCH_SIZE,
    out_path="/content/qwen25vl_test_2k.npy",
)

y_train_small = y_train
y_test_small = y_test


clf = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=6,
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

clf.fit(
    X_train_emb,
    y_train_small,
    eval_set=[(X_test_emb, y_test_small)],
    eval_metric="multi_logloss",
    callbacks=[
        lgb.early_stopping(5),
        lgb.log_evaluation(5),
    ],
)

In [ ]:
LABEL_NAMES = [
    "True",
    "Satire / parody",
    "False connection",
    "Imposter content",
    "Manipulated",
    "Misleading",
]

y_proba = clf.predict_proba(X_test_emb)
y_pred = y_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test_small, y_pred))
print("F1 macro:", f1_score(y_test_small, y_pred, average="macro"))
print("ROC-AUC macro OVR:", roc_auc_score(
    y_test_small,
    y_proba,
    multi_class="ovr",
    average="macro",
))

print(classification_report(
    y_test_small,
    y_pred,
    target_names=LABEL_NAMES,
    digits=4,
))